In [1]:
pip install pyspark

In [3]:
!ls -la /content/

total 16
drwxr-xr-x 1 root root 4096 Nov 18 15:58 .
drwxr-xr-x 1 root root 4096 Nov 22 16:55 ..
drwxr-xr-x 4 root root 4096 Nov 18 15:58 .config
drwxr-xr-x 1 root root 4096 Nov 18 15:58 sample_data


In [5]:
import os
import kagglehub
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, size, split, avg

dataset_dir = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
csv_path = os.path.join(dataset_dir, "IMDB Dataset.csv")
print("Dataset path:", csv_path)

spark = SparkSession.builder \
    .appName("IMDB Movie Reviews Processing") \
    .getOrCreate()

df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

df.printSchema()
df.show(5, truncate=60)

print("Total reviews:", df.count())
df.groupBy("sentiment").count().show()

cleaned_df = df.withColumn(
    "clean_review",
    lower(regexp_replace(col("review"), "<br\\s*/?>", " "))
).withColumn(
    "word_count",
    size(split(col("clean_review"), "\\s+"))
)

cleaned_df.select("sentiment", "word_count", "clean_review").show(5, truncate=50)

cleaned_df.groupBy("sentiment") \
    .agg(avg("word_count").alias("avg_word_count")) \
    .show()

spark.stop()

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Dataset path: /kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv
root
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)

+------------------------------------------------------------+---------+
|                                                      review|sentiment|
+------------------------------------------------------------+---------+
|One of the other reviewers has mentioned that after watch...| positive|
|A wonderful little production. <br /><br />The filming te...| positive|
|I thought this was a wonderful way to spend time on a too...| positive|
|Basically there's a family where a little boy (Jake) thin...| negative|
|Petter Mattei's "Love in the Time of Money" is a visually...| positive|
+------------------------------------------------------------+---------+
only showing top 5 rows
Total reviews: 50000
+---------+-----+
|sentiment|count|
+---------